# 7000LBSBAI - Programming for Business Analytics
## Week 10 - Task: Modelling Spend and Cancellation at FitLife

---

**Module:** Programming for Business Analytics  
**Week:** 10  
**Time:** About 75 minutes, plus optional extensions  
**Topics covered:** Multiple linear regression · Choosing predictors · Logistic regression and odds ratios · Predicted probabilities · Gemini as a reviewer · Getting a notebook ready to submit

---

### How to use this notebook

| Symbol | Meaning |
|--------|---------|
| ✍️ Worked example | Code is already written -> run it and study the output |
| ➡️ Your turn | Write or complete the code in this cell |
| 💬 What does it mean? | Write a short interpretation of your results |
| 🤖 AI activity | A guided activity using Gemini in Colab |
| 🔴 Extension | For students who finish early (optional) |
| 📝 Hint/Tip | Some friendly help and advice |

To run a cell: click on it and press **Shift + Enter**.

---

## ⚠️ BEFORE YOU CONTINUE

### Save a copy of this notebook

Make sure your notebook is saved to your Google Drive:
- **File → Save a copy in Drive**
- Name it: `Week10_Task_FitLife_Regression_Models_YourName.ipynb`

---

### ✨ Unhide generative AI features (keep inline completions turned off)

Go to **Tools** -> **Settings** -> **AI Assistance**

> Uncheck `☐ Show AI-powered inline completions`  
>
> Uncheck `☐ Hide generative AI features`

---
## The scenario

You are finishing your analysis of **FitLife Gyms**. In Weeks 8 and 9 you looked at one or two factors at a time. The membership manager now wants to know:

> ### *"Which factors drive how much members spend, and which factors make members more likely to cancel, once we take everything into account?"*

That calls for **two** models:

| Question | Outcome | Model |
|---|---|---|
| What drives spend? | `monthly_spend` (a number) | **Linear** regression |
| What makes members cancel? | `cancelled_flag` (1/0) | **Logistic** regression |

> 📝 Hints say which **code-along block** or **week** covers each skill. Tables of results are the format to use in your Assessment 2 notebook and report.

---
## Set up: run this cell

This cell loads and cleans FitLife with the Week 8 function, and creates `cancelled_flag` from `cancel_date`, as you did in Week 9.

In [ ]:
# ✍️ Set up: just run this cell
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

# Set the Seaborn chart style
sns.set_theme(style='whitegrid')

# Create a function to clean the raw FitLife member data
def clean_members(data):
    """Clean the FitLife member data (the Week 8 function)."""
    data = data.copy()
    data = data.drop_duplicates()
    data['membership_type'] = data['membership_type'].str.strip().str.title()
    data['monthly_fee'] = pd.to_numeric(data['monthly_fee'].str.replace('£', ''))
    data = data[(data['age'] >= 18) & (data['age'] <= 100)]
    data['visits_per_month'] = data['visits_per_month'].fillna(data['visits_per_month'].median())
    return data

# Import the FitLife Members data
raw = pd.read_csv("https://raw.githubusercontent.com/7000lbsbai/7000LBSBAI/main/datasets/fitlife_members.csv")

# Run the clean_members function on the raw dataset
df = clean_members(raw)

# The outcome, created from the date (Week 9): 1 if the member has a cancel date
df['cancelled_flag'] = np.where(df['cancel_date'].notna(), 1, 0)

# Check all processed correctly
print(f"Data ready: {df.shape[0]} members, {df['cancelled_flag'].mean() * 100:.1f}% cancelled")

---
# Section A: What drives spend?
*about 15 minutes*

In Week 9, visits alone explained only about **24%** of the variation in spend (R² = 0.24). Now include several predictors at once.

> 📝 **Hint:** code-along Block 1. Membership type is a category, so use `C(membership_type)`. The reference group is **Annual** (first alphabetically).

In [ ]:
# ➡️ A.1 Fit a linear model of monthly_spend explained by visits_per_month, classes_per_month, pt_sessions_per_month and C(membership_type). Show a table of the coefficients, p-values and 95% confidence intervals, and print the adjusted R²


### 💬 What does it mean?

- Explain the coefficients for personal training and for Monthly Flex in plain English.
- Why is the coefficient for visits smaller here (£1.85) than in Week 9 (£2.67)?
- How much better is this model than the Week 9 model?

**✏️ Your interpretation (double-click to edit):**

*Write your interpretation here.*

---
# Section B: Choosing predictors
*about 10 minutes*

Each membership type has a standard fee, and some members pay a discounted fee. So `monthly_fee` is **largely decided by** `membership_type`. What happens if you include both?

> 📝 **Hint:** code-along Block 3.

In [ ]:
# ➡️ B.1 Fit the A.1 model again with monthly_fee added. Print the coefficients and p-values. What happens to the membership type coefficients?


### ✏️ B.2 What happened, and what would you do?

- Why did the membership type coefficients shrink and stop being significant when the fee was added?
- Which version would you report, and why?
- A colleague suggests adding `cancel_date` to a model of **cancellation**. Why would that be a mistake?

**Your answer (double-click to edit):**

*Write your answer here.*

---
# Section C: What makes members cancel?
*about 20 minutes*

Now create a new model **cancellation**, a yes/no outcome. Use these predictors: `visits_per_month`, `classes_per_month`, `distance_km`, `age`, `C(membership_type)` and `C(home_gym)`.

> 📝 **Hint:** code-along Block 2. Use `smf.logit(...)` and `.fit(disp=0)`, then `np.exp()` for odds ratios.

In [ ]:
# ➡️ C.1 Fit the logistic model. Show a table of odds ratios with 95% confidence intervals and p-values, without the intercept, rounded to 3 decimal places


### ➡️ C.2 Predicted chances of cancelling

Managers understand **chances** more easily than odds ratios. Copy this table of four example members into the next cell, then add a column with each member's predicted chance of cancelling (%).

```python
members = pd.DataFrame({
    'description': ['Monthly Flex, rarely visits, lives far away', 'Monthly Flex, visits often, lives nearby',
                    'Annual, rarely visits, lives far away', 'Annual, visits often, lives nearby'],
    'visits_per_month': [4, 12, 4, 12],
    'classes_per_month': [1, 3, 1, 3],
    'distance_km': [8, 2, 8, 2],
    'age': [30, 30, 30, 30],
    'membership_type': ['Monthly Flex', 'Monthly Flex', 'Annual', 'Annual'],
    'home_gym': ['City Centre', 'City Centre', 'City Centre', 'City Centre']
})
```

> 📝 **Hint:** code-along Block 2, step 3: `cancel_model.predict(members) * 100`.

In [ ]:
# ➡️ C.2 The predicted chance of cancelling for each example member


### 💬 What does it mean?

- Which factors increase the chance of cancelling, and which reduce it? Use the odds ratios.
- Which factors are **not** significant once everything is taken into account?
- Use the predicted chances to explain the results to the manager in one or two sentences.

**✏️ Your interpretation (double-click to edit):**

*Write your interpretation here.*

---
# Section D: Is a straight line right for age?
*about 10 minutes*

In C.1, age was **not quite significant** (p = 0.052). But in Week 8 you found a **U-shape**: the **youngest** and **oldest** members cancelled most. A single `age` term assumes the effect is a **straight line** (the chance steadily rises or falls with age), so it can miss a U-shape.

Replace `age` with **age bands**, and choose the **35-49** band as the reference group (it had the lowest cancellation rate in Week 8):

```python
df['age_band'] = pd.cut(df['age'], bins=[17, 24, 34, 49, 100], labels=['18-24', '25-34', '35-49', '50+'])
```

To choose a reference group yourself, use `Treatment()` inside `C()`:

```python
C(age_band, Treatment('35-49'))
```

In [ ]:
# ➡️ D.1 Create age_band, then fit the C.1 model with C(age_band, Treatment('35-49')) instead of age. Print the odds ratios and p-values for the age bands only


> 📝 The formula uses **double** quotes on the outside, because `'35-49'` already uses single quotes.

### 💬 What does it mean?

- What does the model with age bands show that the straight-line model missed?
- What does this tell you about choosing the right form for a predictor in your own models?

**✏️ Your interpretation (double-click to edit):**

*Write your interpretation here.*

---
# Section E: 🤖 Gemini as a reviewer
*about 15 minutes*

This week Gemini **reviews** code that already works. You decide which suggestions to use.

### The activity

1. Choose **one** of your own cells from this notebook: **C.1** (the cancellation model) is a good choice.
2. Copy it into the **Gemini** chat (✨), with this prompt:

> *"Review this Python code for readability. Suggest improvements to variable names, comments and how the output is presented. Do not change what it calculates. Explain each suggestion."*

3. Fill in the table below: which suggestions did you **accept** or **reject**, and **why**?
4. Write your improved version in the code cell below, and **check the numbers are unchanged**.

| Gemini's suggestion | Accept or reject? | Why? |
|---|---|---|
| | | |
| | | |
| | | |

In [ ]:
# 🤖 E.1 Your reviewed version (it must give the same numbers)


> 📝 **Checking ideas:** compare one or two odds ratios with C.1, or use `.equals()` to compare two tables.

### 📓 E.2 Add this review to your AI log

Record what you asked, what Gemini suggested, **what you accepted or rejected and why**, and how you checked the result. Examples like this make a strong AI reflection in Assessment 2.

---
# Section F: Get your notebook ready to submit
*about 5 minutes*

Your Assessment 2 notebook must **run from top to bottom without errors**. Practise on this notebook now.

### F.1 Restart and run everything

1. Go to **Runtime → Restart session and run all** (in some versions of Colab: *Restart and run all*).
2. Wait for every cell to finish, then scroll through. **Did every cell run without an error?**

If a cell fails after a restart, it usually means it depends on something you ran **out of order**, or on a cell you have since deleted.

### ✅ The submission checklist (use it for Assessment 2)

- ☐ **Restart session and run all** works, with no errors
- ☐ Every section has a **markdown heading**, and the notebook reads as a document
- ☐ Every chart has a **title** and **labelled axes**, with an interpretation cell beneath it
- ☐ Every output (tables, statistics, models) is **interpreted**: no numbers left uncommented
- ☐ Code has **comments** saying what each block does and why
- ☐ No leftover **test cells**, errors or unused columns
- ☐ Charts you need for the report are **saved** (`plt.savefig()`) and downloaded
- ☐ Download it: **File → Download → Download .ipynb**

### ✏️ F.2 Which model for which question?

- In your own words, how do you decide whether to use linear or logistic regression?
- What would you check **before** choosing the predictors for a model?

**Your answer (double-click to edit):**

*Write your answer here.*

---
## 🔴 Extension 1: a chart of predicted chances

Show how the chance of cancelling falls as visits increase, for each membership type. Create visits from 0 to 25 using `np.arange(0,26)`, hold the other predictors at typical values (the **median** distance, age and classes, and City Centre), and draw one line per membership type.

In [ ]:
# 🔴 Extension 1: predicted chance of cancelling by visits, for each membership type


---
## 🔴 Extension 2: building a model step by step

Use a **loop** to fit linear models of `monthly_spend`, adding one predictor at a time, and print the adjusted R² each time. Which predictor adds the most?

Start with: `['pt_sessions_per_month', 'classes_per_month', 'visits_per_month', 'C(membership_type)', 'distance_km']`

In [ ]:
# 🔴 Extension 2: adding predictors one at a time


---
## ✅ End of Task: Recap

You have built the two main models for an EDA, on real-world-style data:

- **Linear regression** with several predictors, and what "holding the others constant" means
- **Choosing predictors**: overlapping columns, leakage, and checking the shape of a relationship
- **Logistic regression** for a yes/no outcome: odds ratios and predicted chances
- **Gemini as a reviewer**: you decided which suggestions to use, and checked nothing changed
- **A notebook ready to submit**: Restart session and run all, and the submission checklist

---

### 💾 Before you leave

Make sure your notebook is saved to your Google Drive, if you have not done so already:
- **File → Save a copy in Drive**
- Name it: `Week10_Task_FitLife_Regression_Models_YourName.ipynb`

---

### 🔮 Next week

Week 11 focuses on the **business report**: turning your notebook into findings, charts and recommendations for a non-technical reader, and writing your **AI reflection**. Bring your Assessment 2 notebook and your AI log.